# XBRL Cross-Validation Analysis

This notebook demonstrates cross-validation between XBRL financial data and PDF-extracted tables.

## Objectives:
1. Download and parse XBRL files from SEC EDGAR
2. Load PDF-extracted tables from our pipeline
3. Map PDF table labels to XBRL concepts
4. Cross-validate numerical values
5. Analyze discrepancies and their causes


In [ ]:
import sys
import os
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from xbrl_validation import (
    XBRLDownloader, XBRLParser, ConceptMapper, 
    CrossValidator, PDFTableLoader
)

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("XBRL Cross-Validation Analysis")
print("=" * 40)


## 1. Download and Parse XBRL Files


In [ ]:
# Initialize XBRL downloader and parser
downloader = XBRLDownloader(user_agent="Sample Company <sample@example.com>")
parser = XBRLParser()

# Download XBRL files for Apple 2023
company = "Apple"
year = "2023"

print(f"Downloading XBRL files for {company} {year}...")
xbrl_files = downloader.download_xbrl_files(company)
print(f"Downloaded {len(xbrl_files)} XBRL files")

# Parse XBRL files
xbrl_elements = []
for xbrl_file in xbrl_files:
    elements = parser.parse_xbrl_file(xbrl_file)
    xbrl_elements.extend(elements)
    print(f"Parsed {len(elements)} elements from {xbrl_file.name}")

print(f"\nTotal XBRL elements: {len(xbrl_elements)}")


In [ ]:
# Display XBRL elements in a DataFrame
xbrl_data = []
for elem in xbrl_elements:
    xbrl_data.append({
        'Concept': elem.concept,
        'Value': elem.value,
        'Period': elem.period,
        'Context': elem.context_ref,
        'Unit': elem.unit_ref
    })

xbrl_df = pd.DataFrame(xbrl_data)
print("XBRL Elements:")
print(xbrl_df.to_string(index=False))
